# Klasifikasi Status Akademik Mahasiswa
## Implementasi KNN dan Naive Bayes

| | |
|---|---|
| **Nama** | *(isi nama)* |
| **NIM** | *(isi NIM)* |
| **Kelas** | *(isi kelas)* |
| **Dataset** | Student Academic Performance (4.424 data, 37 variabel) |

---

### Deskripsi Tugas

Notebook ini menerapkan proses **Data Science** secara lengkap:
1. Pemahaman, inspeksi, dan penggambaran data
2. Pembersihan data
3. Pemahaman tipe data fitur dan cara penanganannya
4. Implementasi model **K-Nearest Neighbor (KNN)** dan **Naive Bayes**
5. Evaluasi performa model dengan *confusion matrix* dan analisis
6. Eksperimen peningkatan performa

**Target Klasifikasi:** Memprediksi status akademik mahasiswa — **Graduate** (Lulus), **Dropout** (Keluar), atau **Enrolled** (Masih Aktif).

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.preprocessing import (
    LabelEncoder, StandardScaler, MinMaxScaler,
    PowerTransformer, label_binarize
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, cross_val_predict
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier

plt.rcParams['figure.dpi'] = 110

KELAS = ['Dropout', 'Enrolled', 'Graduate']
WARNA = {'Dropout': '#e74c3c', 'Enrolled': '#3498db', 'Graduate': '#2ecc71'}

print('✅ Library berhasil diimport.')

---
## 2. Pemahaman, Inspeksi, dan Penggambaran Data Awal

### 2.1 Memuat Dataset

In [ ]:
df_data = pd.read_csv('data.csv', sep=';')

print(f'Jumlah baris  : {df_data.shape[0]:,}')
print(f'Jumlah kolom  : {df_data.shape[1]}')
print(f'Kolom target  : Status')
print()
df_data.head()

### 2.2 Inspeksi Tipe Data dan Informasi Umum

In [ ]:
print('=== Informasi Dataset ===')
df_data.info()
print()
print('=== Statistik Deskriptif (5 fitur pertama) ===')
df_data.iloc[:, :5].describe().round(2)

### 2.3 Pemahaman Tipe Data Fitur

Dataset ini terdiri dari beberapa kelompok fitur:

| Kelompok | Contoh Fitur | Tipe | Penanganan |
|----------|-------------|------|------------|
| **Demografi** | `Gender`, `Marital_status`, `Age_at_enrollment` | Kategorik/Numerik | Normalisasi / Label Encoding |
| **Akademik Sebelumnya** | `Previous_qualification_grade`, `Admission_grade` | Numerik Kontinu | Normalisasi, cek outlier |
| **Performa Semester** | `Curricular_units_Xst_sem_grade/approved/enrolled` | Numerik Kontinu | Normalisasi, cek outlier, *feature engineering* |
| **Ekonomi Makro** | `Unemployment_rate`, `Inflation_rate`, `GDP` | Numerik Kontinu | Normalisasi |
| **Keuangan** | `Scholarship_holder`, `Tuition_fees_up_to_date`, `Debtor` | Biner (0/1) | Tidak perlu scaling khusus |
| **Target** | `Status` | Kategorik (3 kelas) | Label Encoding |

In [ ]:
# Rangkuman tipe data
print('=== Rangkuman Tipe Data ===')
print(f'Fitur bertipe int64   : {(df_data.dtypes == "int64").sum()}')
print(f'Fitur bertipe float64 : {(df_data.dtypes == "float64").sum()}')
print(f'Fitur bertipe object  : {(df_data.dtypes == "object").sum()}')
print()

# Distribusi kelas target
distribusi_kelas = df_data['Status'].value_counts()
print('=== Distribusi Kelas Target ===')
for kelas, jumlah in distribusi_kelas.items():
    pct = jumlah / len(df_data) * 100
    bar = '█' * int(pct / 2)
    print(f'  {kelas:<12}: {jumlah:>5,} data ({pct:>5.1f}%)  {bar}')

### 2.4 Visualisasi Data Awal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Distribusi kelas target
warna_list = [WARNA[k] for k in distribusi_kelas.index]
axes[0].bar(distribusi_kelas.index, distribusi_kelas.values,
            color=warna_list, edgecolor='gray', width=0.5)
for i, (kls, jml) in enumerate(distribusi_kelas.items()):
    axes[0].text(i, jml + 30, f'{jml:,}', ha='center',
                 fontsize=11, fontweight='bold')
axes[0].set_title('Distribusi Kelas Target', fontweight='bold')
axes[0].set_ylabel('Jumlah Data')

# 2. Rata-rata nilai akademik per kelas
rata_nilai = df_data.groupby('Status')[
    'Curricular_units_1st_sem_grade'].mean().reindex(KELAS)
axes[1].bar(rata_nilai.index, rata_nilai.values,
            color=[WARNA[k] for k in rata_nilai.index],
            edgecolor='gray', width=0.5)
for i, v in enumerate(rata_nilai.values):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center',
                 fontsize=10, fontweight='bold')
axes[1].set_title('Rata-rata Nilai Sem. 1 per Kelas', fontweight='bold')
axes[1].set_ylabel('Rata-rata Nilai')

# 3. Distribusi usia
for kls in KELAS:
    axes[2].hist(df_data[df_data['Status'] == kls]['Age_at_enrollment'],
                 bins=20, alpha=0.6, color=WARNA[kls], label=kls, edgecolor='none')
axes[2].set_title('Distribusi Usia saat Mendaftar', fontweight='bold')
axes[2].set_xlabel('Usia (tahun)')
axes[2].set_ylabel('Frekuensi')
axes[2].legend(fontsize=9)

plt.suptitle('Eksplorasi Data Awal', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap korelasi (fitur numerik kontinu)
fitur_korelasi = [
    'Admission_grade', 'Age_at_enrollment',
    'Curricular_units_1st_sem_grade', 'Curricular_units_1st_sem_approved',
    'Curricular_units_2nd_sem_grade', 'Curricular_units_2nd_sem_approved',
    'Unemployment_rate', 'Inflation_rate', 'GDP'
]
matriks_korel = df_data[fitur_korelasi].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(matriks_korel, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Matriks Korelasi Fitur Utama', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 3. Pembersihan Data (Data Cleaning)

### 3.1 Pengecekan Duplikat dan Missing Value

In [ ]:
jumlah_duplikat = df_data.duplicated().sum()
jumlah_hilang   = df_data.isnull().sum().sum()

print(f'Jumlah baris duplikat : {jumlah_duplikat}')
print(f'Jumlah nilai hilang   : {jumlah_hilang}')

if jumlah_duplikat > 0:
    df_data = df_data.drop_duplicates().reset_index(drop=True)
    print(f'→ Duplikat dihapus. Sisa data: {len(df_data):,}')
else:
    print('→ Data sudah bersih: tidak ada duplikat maupun nilai hilang.')

### 3.2 Penanganan Outlier (IQR Winsorizing)

Outlier ditangani dengan metode **Winsorizing (IQR Capping)**: nilai yang melewati batas [Q1 - 1.5×IQR, Q3 + 1.5×IQR] dikembalikan ke batas tersebut. Metode ini dipilih agar **jumlah baris tidak berkurang**.

In [ ]:
kolom_kontinu = [
    'Previous_qualification_grade', 'Admission_grade', 'Age_at_enrollment',
    'Curricular_units_1st_sem_credited', 'Curricular_units_1st_sem_enrolled',
    'Curricular_units_1st_sem_evaluations', 'Curricular_units_1st_sem_approved',
    'Curricular_units_1st_sem_grade', 'Curricular_units_1st_sem_without_evaluations',
    'Curricular_units_2nd_sem_credited', 'Curricular_units_2nd_sem_enrolled',
    'Curricular_units_2nd_sem_evaluations', 'Curricular_units_2nd_sem_approved',
    'Curricular_units_2nd_sem_grade', 'Curricular_units_2nd_sem_without_evaluations',
    'Unemployment_rate', 'Inflation_rate', 'GDP'
]

df_bersih = df_data.copy()
total_di_cap = 0

for kolom in kolom_kontinu:
    Q1 = df_bersih[kolom].quantile(0.25)
    Q3 = df_bersih[kolom].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas  = Q3 + 1.5 * IQR
    n_outlier = ((df_bersih[kolom] < batas_bawah) | (df_bersih[kolom] > batas_atas)).sum()
    total_di_cap += n_outlier
    df_bersih[kolom] = df_bersih[kolom].clip(lower=batas_bawah, upper=batas_atas)

print(f'Metode penanganan outlier : IQR Winsorizing (Capping)')
print(f'Total nilai yang di-cap   : {total_di_cap}')
print(f'Jumlah baris tetap        : {len(df_bersih):,}')

---
## 4. Pra-pemrosesan Data (Preprocessing)

### 4.1 Rekayasa Fitur (Feature Engineering)

Fitur baru dibuat untuk menangkap pola yang tidak terlihat dari fitur tunggal.

In [ ]:
# Tingkat kelulusan per semester
df_bersih['Tingkat_Lulus_Sem1'] = np.where(
    df_bersih['Curricular_units_1st_sem_enrolled'] > 0,
    df_bersih['Curricular_units_1st_sem_approved'] /
    df_bersih['Curricular_units_1st_sem_enrolled'], 0)

df_bersih['Tingkat_Lulus_Sem2'] = np.where(
    df_bersih['Curricular_units_2nd_sem_enrolled'] > 0,
    df_bersih['Curricular_units_2nd_sem_approved'] /
    df_bersih['Curricular_units_2nd_sem_enrolled'], 0)

# Total mata kuliah lulus
df_bersih['Total_MK_Lulus'] = (
    df_bersih['Curricular_units_1st_sem_approved'] +
    df_bersih['Curricular_units_2nd_sem_approved'])

# Rata-rata nilai dua semester
df_bersih['Rata_Nilai'] = (
    df_bersih['Curricular_units_1st_sem_grade'] +
    df_bersih['Curricular_units_2nd_sem_grade']) / 2

# Performa terbobot
df_bersih['Performa_Sem1'] = (
    df_bersih['Tingkat_Lulus_Sem1'] *
    df_bersih['Curricular_units_1st_sem_grade'])
df_bersih['Performa_Sem2'] = (
    df_bersih['Tingkat_Lulus_Sem2'] *
    df_bersih['Curricular_units_2nd_sem_grade'])

# Stabilitas keuangan
df_bersih['Stabilitas_Keuangan'] = (
    df_bersih['Scholarship_holder'].astype(int) +
    df_bersih['Tuition_fees_up_to_date'].astype(int) -
    df_bersih['Debtor'].astype(int))

# Kemajuan nilai antar semester
df_bersih['Kemajuan_Nilai'] = (
    df_bersih['Curricular_units_2nd_sem_grade'] -
    df_bersih['Curricular_units_1st_sem_grade'])

fitur_baru = ['Tingkat_Lulus_Sem1', 'Tingkat_Lulus_Sem2', 'Total_MK_Lulus',
              'Rata_Nilai', 'Performa_Sem1', 'Performa_Sem2',
              'Stabilitas_Keuangan', 'Kemajuan_Nilai']

print(f'✅ {len(fitur_baru)} fitur baru berhasil dibuat:')
for f in fitur_baru:
    print(f'   • {f}')

### 4.2 Seleksi Fitur (Mutual Information)

Fitur-fitur yang tidak relevan atau memiliki skor *Mutual Information* rendah dihilangkan untuk mengurangi noise dan meningkatkan performa model.

In [ ]:
le_temp = LabelEncoder()
y_temp  = le_temp.fit_transform(df_bersih['Status'])

kolom_hapus = ['Status', 'Application_order', 'Course', 'Nacionality']
X_semua = df_bersih.drop(columns=kolom_hapus)

skor_mi = mutual_info_classif(X_semua, y_temp, random_state=42)
df_mi   = pd.DataFrame({'Fitur': X_semua.columns, 'Skor MI': skor_mi})
df_mi   = df_mi.sort_values('Skor MI', ascending=False).reset_index(drop=True)

AMBANG_MI      = 0.02
fitur_terpilih = df_mi[df_mi['Skor MI'] >= AMBANG_MI]['Fitur'].tolist()
fitur_dibuang  = df_mi[df_mi['Skor MI'] <  AMBANG_MI]['Fitur'].tolist()

print(f'Metode seleksi : Mutual Information (ambang MI ≥ {AMBANG_MI})')
print(f'Fitur semula   : {X_semua.shape[1]}')
print(f'Fitur terpilih : {len(fitur_terpilih)}')
print(f'Fitur dibuang  : {len(fitur_dibuang)}')
print()
print('10 Fitur Paling Informatif:')
print(df_mi.head(10).to_string(index=False))

In [ ]:
# Visualisasi kepentingan fitur
fig, ax = plt.subplots(figsize=(10, 7))
warna_bar = ['#2196F3' if s >= AMBANG_MI else '#cccccc' for s in df_mi['Skor MI']]
ax.barh(df_mi['Fitur'][::-1], df_mi['Skor MI'][::-1],
        color=warna_bar[::-1], height=0.7)
ax.axvline(AMBANG_MI, color='red', linestyle='--', linewidth=1.5,
           label=f'Ambang MI = {AMBANG_MI}')
ax.set_xlabel('Skor Mutual Information', fontsize=11)
ax.set_title('Kepentingan Fitur (Mutual Information)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 4.3 Encoding Label Target & Pembagian Data

In [ ]:
# Encoding target: Dropout=0, Enrolled=1, Graduate=2
enkoder_label = LabelEncoder()
y_data = enkoder_label.fit_transform(df_bersih['Status'])
X_data = df_bersih[fitur_terpilih]

peta_kelas = dict(zip(enkoder_label.classes_,
                       enkoder_label.transform(enkoder_label.classes_)))
print('Peta Encoding Label:')
for kls, kode in sorted(peta_kelas.items(), key=lambda x: x[1]):
    print(f'  {kls} → {kode}')

# Pembagian data: 80% latih, 20% uji (Stratified)
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data)

print()
print(f'Data latih  : {len(X_latih):,} baris')
print(f'Data uji    : {len(X_uji):,}  baris')
print(f'Jumlah fitur: {X_latih.shape[1]}')

### 4.4 Normalisasi Data

StandardScaler di-*fit* **hanya pada data latih** untuk mencegah *data leakage*.

In [ ]:
normalisasi = StandardScaler()
X_latih_norm = pd.DataFrame(
    normalisasi.fit_transform(X_latih), columns=fitur_terpilih)
X_uji_norm   = pd.DataFrame(
    normalisasi.transform(X_uji), columns=fitur_terpilih)

X_semua_norm = pd.concat([X_latih_norm, X_uji_norm], ignore_index=True)
y_semua      = np.concatenate([y_latih, y_uji])

print('✅ Normalisasi selesai (StandardScaler, fit hanya pada data latih).')
print(f'   Mean fitur pertama setelah scaling: {X_latih_norm.iloc[:, 0].mean():.6f}')
print(f'   Std  fitur pertama setelah scaling: {X_latih_norm.iloc[:, 0].std():.6f}')

---
## 5. Implementasi Model

Dua algoritma yang diimplementasikan:
- **K-Nearest Neighbor (KNN)** — berbasis jarak, tidak berasumsi distribusi tertentu
- **Gaussian Naive Bayes** — berbasis probabilitas, mengasumsikan distribusi normal

### 5.1 K-Nearest Neighbor (KNN)

In [ ]:
# ── Cari nilai K optimal ──────────────────────────────────────────────────────
kfold_5    = StratifiedKFold(5, shuffle=True, random_state=42)
nilai_k    = range(1, 26)
skor_cv_k  = []

for k in nilai_k:
    model_k = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    skor = cross_val_score(model_k, X_semua_norm, y_semua,
                           cv=kfold_5, scoring='accuracy').mean()
    skor_cv_k.append(skor)

k_terbaik = nilai_k[np.argmax(skor_cv_k)]
print(f'Nilai K terbaik: {k_terbaik}  (CV Akurasi: {max(skor_cv_k)*100:.2f}%)')

# Visualisasi pemilihan K
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(nilai_k, [s*100 for s in skor_cv_k], 'o-', color='#2196F3',
        linewidth=2, markersize=6)
ax.axvline(k_terbaik, color='red', linestyle='--', linewidth=2,
           label=f'K terbaik = {k_terbaik}')
ax.set_xlabel('Nilai K', fontsize=11)
ax.set_ylabel('Akurasi CV (%)', fontsize=11)
ax.set_title('Pemilihan Nilai K Optimal (5-Fold CV)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xticks(nilai_k)
plt.tight_layout()
plt.show()

In [ ]:
# ── Latih KNN dengan K terbaik ────────────────────────────────────────────────
kfold_10 = StratifiedKFold(10, shuffle=True, random_state=42)

model_knn = KNeighborsClassifier(n_neighbors=k_terbaik, metric='euclidean')
model_knn.fit(X_latih_norm, y_latih)

prediksi_knn = model_knn.predict(X_uji_norm)
akurasi_knn  = accuracy_score(y_uji, prediksi_knn)
cv_knn       = cross_val_score(model_knn, X_semua_norm, y_semua, cv=kfold_10)

print('=== MODEL KNN (K-Nearest Neighbor) ===')
print(f'Nilai K       : {k_terbaik}')
print(f'Akurasi Uji   : {akurasi_knn*100:.2f}%')
print(f'CV 10-Fold    : {cv_knn.mean()*100:.2f}% ± {cv_knn.std()*100:.2f}%')
print()
print('Laporan Klasifikasi:')
print(classification_report(y_uji, prediksi_knn, target_names=KELAS))

### 5.2 Gaussian Naive Bayes (Dasar)

In [ ]:
model_gnb = GaussianNB()
model_gnb.fit(X_latih_norm, y_latih)

prediksi_gnb = model_gnb.predict(X_uji_norm)
akurasi_gnb  = accuracy_score(y_uji, prediksi_gnb)
cv_gnb       = cross_val_score(model_gnb, X_semua_norm, y_semua, cv=kfold_10)

print('=== MODEL NAIVE BAYES (GaussianNB Dasar) ===')
print(f'Akurasi Uji   : {akurasi_gnb*100:.2f}%')
print(f'CV 10-Fold    : {cv_gnb.mean()*100:.2f}% ± {cv_gnb.std()*100:.2f}%')
print()
print('Laporan Klasifikasi:')
print(classification_report(y_uji, prediksi_gnb, target_names=KELAS))

---
## 6. Evaluasi Model — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pasangan = [
    ('KNN (K=' + str(k_terbaik) + ')', prediksi_knn, akurasi_knn),
    ('Gaussian Naive Bayes', prediksi_gnb, akurasi_gnb)
]

for ax, (judul, prediksi, akurasi) in zip(axes, pasangan):
    cm   = confusion_matrix(y_uji, prediksi)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(KELAS, fontsize=9)
    ax.set_yticklabels(KELAS, fontsize=9)
    ax.set_xlabel('Prediksi', fontsize=10)
    ax.set_ylabel('Aktual', fontsize=10)
    ax.set_title(f'{judul}\nAkurasi: {akurasi*100:.2f}%',
                 fontweight='bold', fontsize=11)

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]}\n({cm_n[i,j]*100:.1f}%)',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if cm_n[i,j] > 0.4 else 'black')
    # Highlight diagonal
    for k in range(3):
        ax.add_patch(plt.Rectangle((k-0.5, k-0.5), 1, 1, fill=False,
                                    edgecolor='#2ecc71', linewidth=2))

plt.suptitle('Confusion Matrix — Perbandingan KNN vs Naive Bayes',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print()
print('Interpretasi Confusion Matrix:')
print('  • Diagonal (hijau) = prediksi BENAR')
print('  • Di luar diagonal = prediksi SALAH (mis-klasifikasi)')
print('  • Angka pertama = jumlah data, angka dalam kurung = persentase per kelas aktual')

---
## 7. Eksperimen Peningkatan Performa

Dua strategi peningkatan diuji:

| Strategi | Alasan |
|----------|--------|
| **KNN + Distance Weighting** | Tetangga lebih dekat diberi bobot lebih besar |
| **NB + PowerTransformer + Tuning** | Memperbaiki asumsi distribusi Gaussian, optimasi *var_smoothing* |

### 7.1 KNN — Eksperimen Peningkatan

In [ ]:
# Variasi 1: bobot berdasarkan jarak
model_knn_w = KNeighborsClassifier(
    n_neighbors=k_terbaik, metric='euclidean', weights='distance')
model_knn_w.fit(X_latih_norm, y_latih)
prediksi_knn_w = model_knn_w.predict(X_uji_norm)
akurasi_knn_w  = accuracy_score(y_uji, prediksi_knn_w)
cv_knn_w       = cross_val_score(
    model_knn_w, X_semua_norm, y_semua, cv=kfold_10).mean()

# Variasi 2: scaling MinMax + K terbaik ulang
normalisasi_mm = MinMaxScaler()
X_latih_mm = pd.DataFrame(
    normalisasi_mm.fit_transform(X_latih), columns=fitur_terpilih)
X_uji_mm   = pd.DataFrame(
    normalisasi_mm.transform(X_uji), columns=fitur_terpilih)
X_semua_mm = pd.concat([X_latih_mm, X_uji_mm], ignore_index=True)

skor_cv_k_mm = []
for k in nilai_k:
    m = KNeighborsClassifier(n_neighbors=k, weights='distance')
    s = cross_val_score(m, X_semua_mm, y_semua,
                        cv=kfold_5, scoring='accuracy').mean()
    skor_cv_k_mm.append(s)
k_terbaik_mm = nilai_k[np.argmax(skor_cv_k_mm)]

model_knn_mm = KNeighborsClassifier(
    n_neighbors=k_terbaik_mm, weights='distance')
model_knn_mm.fit(X_latih_mm, y_latih)
prediksi_knn_mm = model_knn_mm.predict(X_uji_mm)
akurasi_knn_mm  = accuracy_score(y_uji, prediksi_knn_mm)
cv_knn_mm       = cross_val_score(
    model_knn_mm, X_semua_mm, y_semua, cv=kfold_10).mean()

print('=== EKSPERIMEN KNN ===')
print(f'{"Varian":<35} {"Akurasi Uji":>12} {"CV 10-Fold":>12}')
print('-'*62)
print(f'{"KNN (standar, StandardScaler)":<35} {akurasi_knn*100:>11.2f}% {cv_knn.mean()*100:>11.2f}%')
print(f'{"KNN (bobot jarak, StandardScaler)":<35} {akurasi_knn_w*100:>11.2f}% {cv_knn_w*100:>11.2f}%')
print(f'{"KNN (bobot jarak, MinMaxScaler)":<35} {akurasi_knn_mm*100:>11.2f}% {cv_knn_mm*100:>11.2f}%')

### 7.2 Naive Bayes — Eksperimen Peningkatan

In [ ]:
# Cari var_smoothing optimal
nilai_vs   = np.logspace(-12, 0, 30)
skor_cv_vs = []

for vs in nilai_vs:
    pipe = Pipeline([
        ('pt',  PowerTransformer(method='yeo-johnson')),
        ('gnb', GaussianNB(var_smoothing=vs))
    ])
    s = cross_val_score(pipe, X_data, y_data,
                        cv=kfold_5, scoring='accuracy').mean()
    skor_cv_vs.append(s)

vs_terbaik = nilai_vs[np.argmax(skor_cv_vs)]
print(f'var_smoothing terbaik: {vs_terbaik:.2e}')

# Model NB yang ditingkatkan: PT + Tuned GNB
pipa_gnb_opt = Pipeline([
    ('pt',  PowerTransformer(method='yeo-johnson')),
    ('gnb', GaussianNB(var_smoothing=vs_terbaik))
])
pipa_gnb_opt.fit(X_latih, y_latih)
prediksi_gnb_opt = pipa_gnb_opt.predict(X_uji)
akurasi_gnb_opt  = accuracy_score(y_uji, prediksi_gnb_opt)
cv_gnb_opt       = cross_val_score(
    pipa_gnb_opt, X_data, y_data, cv=kfold_10).mean()

# Soft Voting Ensemble NB
pipa_1 = Pipeline([('pt', PowerTransformer(method='yeo-johnson')),
                    ('gnb', GaussianNB(var_smoothing=vs_terbaik))])
pipa_2 = Pipeline([('ss', StandardScaler()),
                    ('gnb', GaussianNB(var_smoothing=vs_terbaik))])
pipa_3 = Pipeline([('mm', MinMaxScaler()),
                    ('cnb', ComplementNB())])

model_ensemble_nb = VotingClassifier(
    estimators=[('gnb_pt', pipa_1), ('gnb_ss', pipa_2), ('cnb_mm', pipa_3)],
    voting='soft', weights=[2, 1, 1])
model_ensemble_nb.fit(X_latih, y_latih)
prediksi_ensemble = model_ensemble_nb.predict(X_uji)
akurasi_ensemble  = accuracy_score(y_uji, prediksi_ensemble)
cv_ensemble       = cross_val_score(
    model_ensemble_nb, X_data, y_data, cv=kfold_10).mean()

print()
print('=== EKSPERIMEN NAIVE BAYES ===')
print(f'{"Varian":<40} {"Akurasi Uji":>12} {"CV 10-Fold":>12}')
print('-'*67)
print(f'{"GaussianNB (standar)":<40} {akurasi_gnb*100:>11.2f}% {cv_gnb.mean()*100:>11.2f}%')
print(f'{"GaussianNB + PowerTransformer + Tuning":<40} {akurasi_gnb_opt*100:>11.2f}% {cv_gnb_opt*100:>11.2f}%')
print(f'{"Soft Voting Ensemble NB":<40} {akurasi_ensemble*100:>11.2f}% {cv_ensemble*100:>11.2f}%')

---
## 8. Perbandingan Semua Model & Kurva ROC

In [ ]:
# Tentukan model KNN terbaik dan NB terbaik
knn_hasil = [
    ('KNN (standar)', akurasi_knn,    cv_knn.mean()    ),
    ('KNN (bobot + MM)', akurasi_knn_mm, cv_knn_mm    ),
]
knn_terbaik_nama, akurasi_knn_terbaik, _ = max(knn_hasil, key=lambda x: x[1])

nb_hasil = [
    ('GNB standar',      akurasi_gnb,     cv_gnb.mean()   ),
    ('GNB + PT + Tuning',akurasi_gnb_opt, cv_gnb_opt      ),
    ('NB Ensemble',      akurasi_ensemble,cv_ensemble      ),
]
nb_terbaik_nama, akurasi_nb_terbaik, _ = max(nb_hasil, key=lambda x: x[1])

# Tabel ringkasan
print('='*65)
print('  RINGKASAN PERBANDINGAN SEMUA MODEL')
print('='*65)
print(f'{"Model":<40} {"Akurasi Uji":>12} {"Delta dari GNB":>14}')
print('-'*65)

semua_model = [
    ('KNN (standar)',          akurasi_knn,     cv_knn.mean()   ),
    ('KNN (bobot jarak)',      akurasi_knn_w,   cv_knn_w        ),
    ('KNN (bobot + MinMax)',   akurasi_knn_mm,  cv_knn_mm       ),
    ('GaussianNB (standar) ← baseline', akurasi_gnb, cv_gnb.mean()),
    ('GNB + PowerTransformer', akurasi_gnb_opt, cv_gnb_opt      ),
    ('Soft Voting Ensemble NB',akurasi_ensemble,cv_ensemble      ),
]

for nama, acc, _ in semua_model:
    delta = acc - akurasi_gnb
    sign  = '+' if delta >= 0 else ''
    print(f'{nama:<40} {acc*100:>11.2f}% {sign}{delta*100:>11.2f}%')

print()
print(f'  KNN Terbaik : {knn_terbaik_nama} ({akurasi_knn_terbaik*100:.2f}%)')
print(f'  NB  Terbaik : {nb_terbaik_nama}  ({akurasi_nb_terbaik*100:.2f}%)')
print('='*65)

In [ ]:
# Visualisasi perbandingan akurasi
nama_model  = [r[0].replace(' ← baseline','') for r in semua_model]
akurasi_all = [r[1]*100 for r in semua_model]

warna_cols = ['#3498db','#3498db','#3498db','#e74c3c','#2ecc71','#2ecc71']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(nama_model, akurasi_all, color=warna_cols, edgecolor='gray', height=0.6)
for bar, val in zip(bars, akurasi_all):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', ha='left', fontsize=10, fontweight='bold')
ax.axvline(akurasi_gnb*100, color='red', linestyle='--', linewidth=2,
           label=f'Baseline GNB ({akurasi_gnb*100:.2f}%)')
ax.set_xlabel('Akurasi (%)', fontsize=11)
ax.set_title('Perbandingan Akurasi Semua Model', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(55, 82)
# Legend warna
from matplotlib.patches import Patch
legend_elem = [Patch(facecolor='#3498db', label='KNN'),
               Patch(facecolor='#e74c3c', label='NB Baseline'),
               Patch(facecolor='#2ecc71', label='NB Ditingkatkan')]
ax.legend(handles=legend_elem, fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Kurva ROC — KNN terbaik vs NB terbaik
if knn_terbaik_nama == 'KNN (standar)':
    prob_knn = model_knn.predict_proba(X_uji_norm)
elif knn_terbaik_nama == 'KNN (bobot + MM)':
    prob_knn = model_knn_mm.predict_proba(X_uji_mm)
else:
    prob_knn = model_knn_w.predict_proba(X_uji_norm)

if nb_terbaik_nama == 'NB Ensemble':
    prob_nb = model_ensemble_nb.predict_proba(X_uji)
elif nb_terbaik_nama == 'GNB + PT + Tuning':
    prob_nb = pipa_gnb_opt.predict_proba(X_uji)
else:
    prob_nb = model_gnb.predict_proba(X_uji_norm)

y_uji_biner = label_binarize(y_uji, classes=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (judul, prob) in zip(axes, [
    (f'ROC — KNN ({knn_terbaik_nama})', prob_knn),
    (f'ROC — NB  ({nb_terbaik_nama})',  prob_nb)
]):
    for i, kls in enumerate(KELAS):
        fpr, tpr, _ = roc_curve(y_uji_biner[:, i], prob[:, i])
        auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2,
                color=list(WARNA.values())[i],
                label=f'{kls} (AUC={auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(judul, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

plt.suptitle('Kurva ROC (One-vs-Rest)', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Analisis Performa Model

In [ ]:
# F1-Score per kelas: KNN vs NB terbaik
if knn_terbaik_nama == 'KNN (standar)':
    pred_knn_terbaik = prediksi_knn
elif knn_terbaik_nama == 'KNN (bobot + MM)':
    pred_knn_terbaik = prediksi_knn_mm
else:
    pred_knn_terbaik = prediksi_knn_w

if nb_terbaik_nama == 'NB Ensemble':
    pred_nb_terbaik = prediksi_ensemble
elif nb_terbaik_nama == 'GNB + PT + Tuning':
    pred_nb_terbaik = prediksi_gnb_opt
else:
    pred_nb_terbaik = prediksi_gnb

f1_knn = f1_score(y_uji, pred_knn_terbaik, average=None)
f1_nb  = f1_score(y_uji, pred_nb_terbaik,  average=None)

x = np.arange(3)
lebar = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - lebar/2, f1_knn*100, lebar,
       label=f'KNN ({knn_terbaik_nama})', color='#3498db', edgecolor='gray')
ax.bar(x + lebar/2, f1_nb*100,  lebar,
       label=f'NB ({nb_terbaik_nama})',  color='#2ecc71', edgecolor='gray')
for i in range(3):
    ax.text(i - lebar/2, f1_knn[i]*100 + 0.5, f'{f1_knn[i]*100:.1f}%',
            ha='center', va='bottom', fontsize=9)
    ax.text(i + lebar/2, f1_nb[i]*100  + 0.5, f'{f1_nb[i]*100:.1f}%',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(KELAS)
ax.set_ylabel('F1-Score (%)')
ax.set_title('F1-Score per Kelas — Model Terbaik KNN vs NB',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

---
## 10. Kesimpulan

In [ ]:
f1_knn_macro = f1_score(y_uji, pred_knn_terbaik, average='macro')
f1_nb_macro  = f1_score(y_uji, pred_nb_terbaik,  average='macro')

print('='*65)
print('  KESIMPULAN')
print('='*65)

print(f'''
1. DATASET
   Dataset terdiri dari {len(df_data):,} mahasiswa dengan 3 kelas:
   Graduate ({(df_data["Status"]=="Graduate").sum():,}), \
Dropout ({(df_data["Status"]=="Dropout").sum():,}), \
Enrolled ({(df_data["Status"]=="Enrolled").sum():,}).

2. PEMBERSIHAN DATA
   Tidak ditemukan duplikat maupun missing value. Outlier ditangani
   dengan Winsorizing IQR (total {total_di_cap} nilai di-cap).

3. PREPROCESSING
   Dibuat {len(fitur_baru)} fitur baru. Seleksi MI (ambang 0.02) menghasilkan
   {len(fitur_terpilih)} fitur terpilih. Data dibagi 80:20 (Stratified).

4. HASIL KNN
   Model terbaik : {knn_terbaik_nama}
   Akurasi uji   : {akurasi_knn_terbaik*100:.2f}%
   F1-Macro      : {f1_knn_macro*100:.2f}%

5. HASIL NAIVE BAYES
   Model terbaik : {nb_terbaik_nama}
   Akurasi uji   : {akurasi_nb_terbaik*100:.2f}%
   F1-Macro      : {f1_nb_macro*100:.2f}%

6. ANALISIS
   Kelas Dropout dan Graduate dapat diprediksi dengan baik.
   Kelas Enrolled lebih sulit karena pola fiturnya ambigu
   (mahasiswa belum menyelesaikan studi).
   Peningkatan NB melalui PowerTransformer dan Voting Ensemble
   terbukti memperbaiki distribusi data yang skewed sehingga
   memperkuat asumsi Gaussian NB.
''')
print('='*65)